# IDNA Package Examples

Five examples of the `idna` package: encoding and decoding internationalised domain names (IDNA 2008 / UTS 46).

## Example 1: Basic IDNA encoding and decoding

In [ ]:
import idna

# Basic IDNA encoding (Unicode to ASCII)
unicode_domain = "münchen.de"  # German domain with umlaut
encoded = idna.encode(unicode_domain)
print(f"Unicode domain: {unicode_domain}")
print(f"IDNA encoded: {encoded}")

# Basic IDNA decoding (ASCII to Unicode)
ascii_domain = b"xn--mnchen-3ya.de" 
decoded = idna.decode(ascii_domain)
print(f"\nASCII domain: {ascii_domain}")
print(f"IDNA decoded: {decoded}")

# Verify round-trip consistency
roundtrip = idna.decode(idna.encode(unicode_domain))
print(f"\nRound-trip test: {unicode_domain} == {roundtrip}: {unicode_domain == roundtrip}")

# Test with various international domains
domains = ["例え.テスト", "пример.испытание", "مثال.إختبار"]
for domain in domains:
    try:
        encoded = idna.encode(domain)
        print(f"✓ {domain} -> {encoded}")
    except Exception as e:
        print(f"✗ {domain} -> Error: {e}")
        raise

## Example 2: UTS46 processing with different options

In [ ]:
import idna

# UTS46 processing with different options
test_domain = "ExAmPlE.COM"

# Standard UTS46 processing
processed = idna.uts46_remap(test_domain)
print(f"Original: {test_domain}")
print(f"UTS46 processed: {processed}")

# UTS46 with case folding
processed_fold = idna.uts46_remap(test_domain, std3_rules=True, transitional=False)
print(f"With std3_rules: {processed_fold}")

# Test mixed script domains
mixed_domains = [
    "Bücher.example.com",
    "кафе.example.org", 
    "日本.example.net"
]

for domain in mixed_domains:
    try:
        processed = idna.uts46_remap(domain)
        encoded = idna.encode(processed)
        print(f"✓ {domain} -> {processed} -> {encoded}")
    except Exception as e:
        print(f"✗ {domain} -> Error: {e}")
        raise

# Verify IDNA module attributes
print(f"\nIDNA module has core functions: {all(hasattr(idna, func) for func in ['encode', 'decode', 'uts46_remap'])}")

## Example 3: Error handling and validation

In [ ]:
import idna

# Test various invalid domains
invalid_domains = [
    "",  # Empty domain
    ".example.com",  # Leading dot
    "example..com",  # Double dot
    "a" * 64 + ".com",  # Label too long
    "-example.com",  # Leading hyphen
    "example-.com"  # Trailing hyphen
]

print("Testing invalid domains:")
for domain in invalid_domains:
    try:
        result = idna.encode(domain, strict=True)
    except idna.core.IDNAError as e:
        print(f"✓ Correctly rejected '{domain}': {type(e).__name__}")
    else:
        raise AssertionError(f"idna.encode({domain!r}, strict=True) accepted an invalid domain: {result!r}")

# A trailing dot is NOT invalid: it is the DNS root label of a fully-qualified name, and idna keeps it.
fqdn = idna.encode("example.com.", strict=True)
assert fqdn == b"example.com.", f"unexpected encoding of the fully-qualified name: {fqdn!r}"
print(f"✓ Trailing dot kept as the root label: 'example.com.' -> {fqdn}")

# Test edge cases with valid domains
edge_cases = [
    "a.com",  # Single character
    "1.com",  # Numeric
    "x-y.com",  # Hyphen in middle
    "münchen-café.de"  # Multiple non-ASCII characters
]

print("\nTesting edge cases:")
for domain in edge_cases:
    try:
        encoded = idna.encode(domain)
        decoded = idna.decode(encoded)
        print(f"✓ '{domain}' -> {encoded} -> '{decoded}'")
    except Exception as e:
        print(f"✗ Error with '{domain}': {e}")
        raise

## Example 4: Working with email addresses and URLs

In [ ]:
import idna
import urllib.parse

# Extract and process domains from email addresses
email_addresses = [
    "user@münchen.de",
    "contact@café.paris",
    "info@北京.中国",
    "admin@москва.рф"
]

print("Processing email domains:")
for email in email_addresses:
    try:
        local, domain = email.split('@')
        encoded_domain = idna.encode(domain)
        ascii_email = f"{local}@{encoded_domain.decode('ascii')}"
        print(f"✓ {email} -> {ascii_email}")
    except Exception as e:
        print(f"✗ Error with {email}: {e}")
        raise

# Extract and process domains from URLs
urls = [
    "https://фермер.рф/продукты",
    "http://例え.テスト/パス",
    "https://müller.германия/path"
]

print("\nProcessing URL domains:")
for url in urls:
    try:
        parsed = urllib.parse.urlparse(url)
        domain = parsed.netloc
        encoded_domain = idna.encode(domain)
        ascii_url = url.replace(domain, encoded_domain.decode('ascii'))
        print(f"✓ {url}")
        print(f"  -> {ascii_url}")
    except Exception as e:
        print(f"✗ Error with {url}: {e}")
        raise

# Verify IDNA can handle these use cases
print(f"\nIDNA core functionality verified: {hasattr(idna, 'encode') and hasattr(idna, 'decode')}")

## Example 5: Performance testing and compatibility checks

In [ ]:
import idna
import time

# Performance test with batch processing
test_domains = [
    "münchen.de", "café.fr", "москва.рф", "東京.jp",
    "العربية.السعودية", "ελληνικά.gr", "עברית.il", "हिन्दी.भारत"
] * 10  # 80 domains total

print(f"Performance test with {len(test_domains)} domains:")
start_time = time.time()

encoded_results = []
for domain in test_domains:
    try:
        encoded = idna.encode(domain)
        encoded_results.append(encoded)
    except Exception as e:
        print(f"Error encoding {domain}: {e}")
        raise

encoding_time = time.time() - start_time
print(f"Encoding time: {encoding_time:.4f} seconds")
print(f"Average per domain: {encoding_time/len(test_domains)*1000:.2f} ms")

# Decode them back
start_time = time.time()
decoded_results = []
for encoded in encoded_results:
    try:
        decoded = idna.decode(encoded)
        decoded_results.append(decoded)
    except Exception as e:
        print(f"Error decoding {encoded}: {e}")
        raise

decoding_time = time.time() - start_time
print(f"Decoding time: {decoding_time:.4f} seconds")

# Verify consistency
unique_originals = list(set(test_domains))
unique_decoded = list(set(decoded_results))
consistency_check = len(unique_originals) == len(unique_decoded)
print(f"\nConsistency check: {consistency_check}")
print(f"Original unique domains: {len(unique_originals)}")
print(f"Decoded unique domains: {len(unique_decoded)}")

# Check IDNA module version and capabilities
print(f"\nIDNA module imported successfully: {idna.__name__}")
print(f"Main functions available: {', '.join([func for func in dir(idna) if not func.startswith('_')])}")
print("\nIDNA processing completed successfully!")